## 04 — Embedding-based Models (Semantic Representations)

Goal: Evaluate embedding-based semantic representations for hallucination detection on HaluEval,
using the same data splits and evaluation protocol as prior baselines (Notebooks 02–03).


In [14]:
import sys
from pathlib import Path

# Ensure repo root is in PYTHONPATH when running from notebooks/
ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


In [15]:
import numpy as np
import pandas as pd

from src.data.load_splits import load_splits
from src.features.embeddings import EmbeddingConfig, embed_dataframe
from src.models.embedding_baseline import build_embedding_lr
from src.utils import evaluate_split, metrics_table


In [16]:
train_df, val_df, test_df = load_splits(ROOT)

print(train_df.shape, val_df.shape, test_df.shape)
print(train_df.columns.tolist())


(51605, 6) (6451, 6) (6451, 6)
['id', 'task', 'prompt', 'response', 'label', 'context']


#### Quick sanity checks

In [17]:
print("Label balance (train):")
print(train_df["label"].value_counts(normalize=True))

print("\nResponse length (chars) summary (train):")
print(train_df["response"].astype(str).fillna("").str.len().describe())


Label balance (train):
label
0    0.534929
1    0.465071
Name: proportion, dtype: float64

Response length (chars) summary (train):
count    51605.000000
mean       190.204341
std        197.321469
min          1.000000
25%         44.000000
50%        107.000000
75%        290.000000
max       3381.000000
Name: response, dtype: float64


### MiniLM baseline
Embedding extraction (MiniLM) + cache + MPS

In [18]:
cfg = EmbeddingConfig(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    text_col="response",
    device="mps",                 # Mac acceleration (fallback to "cpu" if needed)
    batch_size=64,
    normalize=True,
    cache_dir="artifacts/embeddings",
)

X_train = embed_dataframe(train_df, cfg, cache_tag="train")
X_val   = embed_dataframe(val_df,   cfg, cache_tag="val")
X_test  = embed_dataframe(test_df,  cfg, cache_tag="test")

y_train = train_df["label"].values
y_val   = val_df["label"].values
y_test  = test_df["label"].values

print(X_train.shape, X_val.shape, X_test.shape)
print("nan?", np.isnan(X_train).any(), "inf?", np.isinf(X_train).any())


(51605, 384) (6451, 384) (6451, 384)
nan? False inf? False


#### Train + Evaluate (MiniLM + LR)

In [19]:
model = build_embedding_lr()
model.fit(X_train, y_train)

m_train = evaluate_split("train_minilm", model, X_train, y_train)
m_val   = evaluate_split("val_minilm",   model, X_val,   y_val)
m_test  = evaluate_split("test_minilm",  model, X_test,  y_test)

m_train, m_val, m_test

/Users/aviv.gross/hallu-detect/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)



train_minilm metrics:
  accuracy = 0.7649
  f1       = 0.7596
  precision= 0.7242
  recall   = 0.7987
  confusion matrix:
[[20304  7301]
 [ 4830 19170]]

val_minilm metrics:
  accuracy = 0.7611
  f1       = 0.7545
  precision= 0.7226
  recall   = 0.7893
  confusion matrix:
[[2542  909]
 [ 632 2368]]

test_minilm metrics:
  accuracy = 0.7675
  f1       = 0.7638
  precision= 0.7239
  recall   = 0.8083
  confusion matrix:
[[2526  925]
 [ 575 2425]]


(SplitMetrics(split='train_minilm', accuracy=0.764925879275264, f1=0.7596441520873373, precision=0.7241887348419025, recall=0.79875),
 SplitMetrics(split='val_minilm', accuracy=0.7611223066191288, f1=0.754500557591206, precision=0.7226121452548062, recall=0.7893333333333333),
 SplitMetrics(split='test_minilm', accuracy=0.7674779104014882, f1=0.7637795275590551, precision=0.7238805970149254, recall=0.8083333333333333))

In [20]:
metrics_table([m_train, m_val, m_test])

,split,accuracy,f1,precision,recall
0,train_minilm,0.764926,0.759644,0.724189,0.798750
1,val_minilm,0.761122,0.754501,0.722612,0.789333
2,test_minilm,0.767478,0.763780,0.723881,0.808333


### Error Analysis (MiniLM + LR)

In [21]:
def get_pred_proba(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, "decision_function"):
        s = model.decision_function(X)
        return (s - s.min()) / (s.max() - s.min() + 1e-9)
    return model.predict(X)

proba_test = get_pred_proba(model, X_test)
pred_test = (proba_test >= 0.5).astype(int)

err_df = test_df.copy()
err_df["pred"] = pred_test
err_df["proba"] = proba_test

fp = err_df[(err_df["label"] == 0) & (err_df["pred"] == 1)].sort_values("proba", ascending=False)
fn = err_df[(err_df["label"] == 1) & (err_df["pred"] == 0)].sort_values("proba", ascending=True)

print("FP:", fp.shape, "FN:", fn.shape)


FP: (925, 8) FN: (575, 8)


#### Display FP / FN examples

In [22]:
def preview_text(s, n=300):
    s = "" if s is None else str(s)
    return s[:n] + ("..." if len(s) > n else "")

def make_error_table(df, n=10):
    out = df.head(n).copy()
    out["prompt_snip"] = out["prompt"].apply(preview_text)
    out["response_snip"] = out["response"].apply(preview_text)
    cols = ["task", "label", "pred", "proba", "prompt_snip", "response_snip"]
    return out[cols]

print("Top False Positives (confident but wrong):")
display(make_error_table(fp, n=10))

print("Top False Negatives (missed hallucinations):")
display(make_error_table(fn, n=10))


Top False Positives (confident but wrong):


,task,label,pred,proba,prompt_snip,response_snip
3770,summarization,0,1,0.977796,"Washington (CNN)Until recently, if you sat in ...",There are now more people of faith who favor m...
1534,dialogue,0,1,0.977392,[Human]: Do you know Alex Morgan? [Assistant]:...,Me neither! LOL
411,dialogue,0,1,0.975498,[Human]: Do you know about the book Perfect Ch...,I'm sorry. Could you repeat that?
920,dialogue,0,1,0.969904,[Human]: Do you know of any movies directed by...,"The movie, Profile, gives a modern twist to th..."
1382,dialogue,0,1,0.966882,[Human]: Do you like Elizabeth Kostova? Any bo...,"She was born in 1964 in New London, Connectic..."
1914,dialogue,0,1,0.965474,[Human]: Could you recommend some books by aut...,"The Martian was released in 2012, I am not sur..."
3172,dialogue,0,1,0.963159,[Human]: I like to know more about The Catcher...,"It was released in 1953, the same year as The ..."
2898,dialogue,0,1,0.959738,[Human]: Could you recommend any good books si...,He wrote The Lightning Thief: The Graphic Nove...
4761,dialogue,0,1,0.959033,[Human]: Could you suggest any movie directed ...,Kirsten Dunst and Eliza Dushku are the stars o...
4644,summarization,0,1,0.954986,Three people have been killed after clubbers s...,Three people were killed and seven were seriou...


Top False Negatives (missed hallucinations):


,task,label,pred,proba,prompt_snip,response_snip
2585,qa,1,0,0.000022,"What album, produced by John Leckie and engine...",OK Computer
1499,qa,1,0,0.000670,When a former broadcaster with ABC Sports retu...,2nd
6203,qa,1,0,0.002895,"Anil V. Kumar, is a television and film direc...",Nagarjuna Akkineni
4330,dialogue,1,0,0.004942,"[Human]: Could you recommend movies like Eat, ...",2007
3660,qa,1,0,0.006830,What is the nickname of the first baseman who ...,Big Poppy
6318,qa,1,0,0.007618,What type of category does Obregonia and Cymbi...,family
5781,qa,1,0,0.015022,"Sir John Piers, 6th Baronet is known for being...",1964
2367,qa,1,0,0.017281,Henry C. McRae served for one season as the he...,private financial assistance
337,qa,1,0,0.018834,What kind of royalty does Hussa bint Ahmed Al ...,Nobility
5691,qa,1,0,0.021528,What extended play in the album by American si...,Fearless


## Error Analysis — MiniLM + Logistic Regression

### False Positives (label = 0, pred = 1)
The model frequently flags non-hallucinatory responses as hallucinations in the following cases:

- **Conversational or non-informative responses**:  
  Short, casual answers (e.g., acknowledgments, refusals, or jokes) that contain little factual content are often classified as hallucinations, despite not making any factual claims.

- **Vague or generic factual statements**:  
  Responses that sound encyclopedic or general but lack concrete details or sources tend to be over-flagged. Semantically weak or abstract phrasing appears similar to hallucination patterns in the embedding space.

- **Task-dependent ambiguity**:  
  False positives are especially common in *dialogue* and *summarization* tasks, where factual grounding is less explicit and “truth” is harder to define, leading the model to confuse stylistic vagueness with factual incorrectness.

---

### False Negatives (label = 1, pred = 0)
The model fails to detect hallucinations primarily in the following scenarios:

- **Short, fluent factual answers**:  
  One-word or very short answers (e.g., names, dates, titles) that are factually incorrect but semantically plausible are often misclassified as non-hallucinatory.

- **Semantically coherent but incorrect responses**:  
  Answers that are linguistically fluent and contextually relevant, yet contain subtle factual errors, are difficult for the model to distinguish from correct responses based on embeddings alone.

- **Question-answering (QA) tasks**:  
  The majority of false negatives occur in QA settings, where hallucinations often manifest as confident but wrong factual assertions rather than stylistic or semantic anomalies.

---

### Interpretation
- **What embeddings capture well**:  
  Semantic coherence, topical relevance, and stylistic similarity to known hallucination patterns, leading to high recall in detecting suspicious or low-content responses.

- **What embeddings miss**:  
  Factual correctness and fine-grained verification, particularly for concise and confident answers that lack explicit uncertainty cues.

- **Hypothesis for improvement**:  
  Combining semantic embeddings with surface-level and uncertainty-aware features (e.g., response length, confidence markers, hedging language, and source presence) can help balance recall and precision by capturing complementary signals that embeddings alone fail to model.


## Hybrid (MiniLM + Numeric)

#### Add numeric features

In [23]:
from src.features import add_numeric_feature_columns, get_numeric_feature_cols
from src.models.embedding_baseline import concat_embeddings_and_numeric

train_df_num = add_numeric_feature_columns(train_df.copy())
val_df_num   = add_numeric_feature_columns(val_df.copy())
test_df_num  = add_numeric_feature_columns(test_df.copy())

num_cols = get_numeric_feature_cols(train_df_num)  # 👈 כאן התיקון
print("Number of numeric features:", len(num_cols))
print("First 10 numeric cols:", num_cols[:10])


Number of numeric features: 10
First 10 numeric cols: ['resp_n_chars', 'resp_n_words', 'resp_n_punct', 'resp_has_multi_excl', 'resp_has_multi_q', 'resp_has_ellipsis', 'resp_n_numbers', 'resp_n_uncertainty', 'resp_punct_per_word', 'resp_numbers_per_word']


#### Build mixed matrices

In [24]:
X_train_num = train_df_num[num_cols].values
X_val_num   = val_df_num[num_cols].values
X_test_num  = test_df_num[num_cols].values

X_train_mix = concat_embeddings_and_numeric(X_train, X_train_num)
X_val_mix   = concat_embeddings_and_numeric(X_val,   X_val_num)
X_test_mix  = concat_embeddings_and_numeric(X_test,  X_test_num)

print(X_train_mix.shape, X_val_mix.shape, X_test_mix.shape)
print("nan?", np.isnan(X_train_mix).any(), "inf?", np.isinf(X_train_mix).any())


(51605, 394) (6451, 394) (6451, 394)
nan? False inf? False


#### Train + Evaluate Hybrid

In [25]:
model_mix = build_embedding_lr()
model_mix.fit(X_train_mix, y_train)

mix_train = evaluate_split("train_minilm+num", model_mix, X_train_mix, y_train)
mix_val   = evaluate_split("val_minilm+num",   model_mix, X_val_mix,   y_val)
mix_test  = evaluate_split("test_minilm+num",  model_mix, X_test_mix,  y_test)

metrics_table([mix_train, mix_val, mix_test])


/Users/aviv.gross/hallu-detect/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)



train_minilm+num metrics:
  accuracy = 0.7830
  f1       = 0.7769
  precision= 0.7444
  recall   = 0.8125
  confusion matrix:
[[20908  6697]
 [ 4501 19499]]

val_minilm+num metrics:
  accuracy = 0.7875
  f1       = 0.7803
  precision= 0.7515
  recall   = 0.8113
  confusion matrix:
[[2646  805]
 [ 566 2434]]

test_minilm+num metrics:
  accuracy = 0.7858
  f1       = 0.7817
  precision= 0.7428
  recall   = 0.8250
  confusion matrix:
[[2594  857]
 [ 525 2475]]


,split,accuracy,f1,precision,recall
0,train_minilm+num,0.783006,0.776914,0.744350,0.812458
1,val_minilm+num,0.787475,0.780253,0.751467,0.811333
2,test_minilm+num,0.785770,0.781744,0.742797,0.825000


In [26]:
rows = [
    {"model": "Embeddings(MiniLM)+LR",  "test_acc": m_test.accuracy,  "test_f1": m_test.f1,  "test_precision": m_test.precision,  "test_recall": m_test.recall},
    {"model": "Embeddings(MiniLM)+num", "test_acc": mix_test.accuracy,"test_f1": mix_test.f1,"test_precision": mix_test.precision,"test_recall": mix_test.recall},
]
df_cmp = pd.DataFrame(rows)
df_cmp


,model,test_acc,test_f1,test_precision,test_recall
0,Embeddings(MiniLM)+LR,0.767478,0.763780,0.723881,0.808333
1,Embeddings(MiniLM)+num,0.785770,0.781744,0.742797,0.825000


### Key findings
- MiniLM embeddings alone achieved ~0.77 test accuracy and ~0.76 F1, with relatively high recall but lower precision.
- Adding numeric surface/uncertainty features improved performance to ~0.79 test accuracy and ~0.78 F1, improving both precision and recall.
- Despite the improvement, the hybrid model still did not surpass the strongest classical baseline (~0.82), suggesting that HaluEval contains strong lexical/surface signals and that semantic representations alone are insufficient.

### Implication
Semantic representations provide complementary signal, but most predictive power appears to come from surface-level or lexical cues, motivating either richer lexical modeling or task-adapted fine-tuning as a next step.
